# Stuff+: from PCA composite to a trained model

Retooled 9/4/2026 (see `docs/dev_log.md`'s entry of the same date). The
original version of this notebook built Stuff+ as an unsupervised PCA
composite: z-score six physics features, let PCA pick the weights, no
outcome data anywhere. That design treated "outlierness" itself as the
signal.

Rereading FanGraphs' own description of their Stuff+ model
(`library.fangraphs.com`'s Stuff+/Location+/Pitching+ primer) put that
assumption in question: their Stuff+ is trained against run value with a
decision-tree model that captures nonlinear physics-to-outcome
relationships, not a variance-maximizing composite. `pitching_plus/scripts/stuff.py`
now works the way `location.py` always has: one `HistGradientBoostingRegressor`
per pitch type, 5-fold out-of-fold cross-validation against
`delta_pitcher_run_exp`, cached under `pitching_plus/models/`. A pitch's own
realized outcome still never leaks into its own score -- that guarantee
didn't change, only how the weights are learned.

This notebook no longer reimplements that machinery inline. The modeling
logic is unit-tested in `pitching_plus/tests/test_stuff.py` and belongs in
one place; this notebook calls the real `add_stuff_plus` directly, the same
way `pitching.ipynb` and `bestpitch.ipynb` already do. What it keeps: the
derivation of the one new piece of domain reasoning (`axis_differential`,
a seam-shifted-wake proxy) and the outcome-validation diagnostic that
originally motivated questioning the old design.

In [1]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pitching_plus").is_dir():
            return candidate
    raise RuntimeError("Could not locate repo root (expected a 'pitching_plus' directory in a parent).")


REPO_ROOT = find_repo_root(Path.cwd())
SCRIPTS_DIR = REPO_ROOT / "pitching_plus" / "scripts"
DATA_PATH = REPO_ROOT / "data" / "MLB_2021-2025.csv"

sys.path.insert(0, str(SCRIPTS_DIR))

import stuff as stuff_mod
from stuff import PITCHER_COL, PITCH_TYPE_COL, SEASON_COL, PLAYER_NAME_COL, JUNK_PITCH_TYPES

## Score the full dataset

`add_stuff_plus` uses the cached models under `pitching_plus/models/`
(`retrain=False`, the default) rather than retraining -- rebuilding them from
scratch takes ~79s on this dataset (`docs/dev_log.md`'s 9/4 entry); scoring
from the cache is a merge against the out-of-fold historical scores, not a
refit.

In [2]:
t0 = time.time()
raw_df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(raw_df):,} raw pitches in {time.time() - t0:.0f}s")

t0 = time.time()
df = stuff_mod.add_stuff_plus(raw_df)
print(f"Scored Stuff+ in {time.time() - t0:.0f}s")

in_scope = df.dropna(subset=["pitch_stuff_plus"])
print(f"{len(in_scope):,} / {len(df):,} pitches have a Stuff+ score")

Loaded 3,565,743 raw pitches in 25s


Scored Stuff+ in 7s


3,525,964 / 3,565,743 pitches have a Stuff+ score


## axis_differential: a seam-shifted-wake proxy

The one new feature in `STUFF_FEATURES`. Statcast already measures two
independent things about a pitch's break: `spin_axis` (the ball's actual
spin direction, from Hawk-Eye camera tracking) and `pfx_x`/`pfx_z` (the
ball's actual observed movement, already gravity-adjusted). Under pure
Magnus-effect physics, a ball's movement direction should point exactly
along its spin axis -- that's the textbook relationship. `axis_differential`
is the angular gap between the two, wrapped to [0, 180]:

    movement_angle = atan2(pfx_x, -pfx_z) mod 360
    axis_differential = |spin_axis - movement_angle| wrapped to [0, 180]

Zero means the ball moved exactly the way its spin says it should ("active
spin" in Trackman's terminology); a large gap means some of the break comes
from somewhere spin alone doesn't explain -- the seam-shifted-wake signature.
No aerodynamic force model is needed (air density, drag/lift coefficients,
Reynolds number): both angles being compared are already measured
quantities, not derived ones.

`_build_v1_features` computes this the same way for every in-scope pitch;
this cell checks the result against known spin-efficiency patterns before
trusting it as a model input.

In [3]:
in_scope_physics = (
    raw_df.loc[~raw_df[PITCH_TYPE_COL].isin(JUNK_PITCH_TYPES), stuff_mod.REQUIRED_COLS + ["p_throws"]]
    .dropna(subset=stuff_mod.REQUIRED_COLS)
    .copy()
)
engineered = stuff_mod._build_v1_features(in_scope_physics)

print("Median axis_differential (degrees) by pitch type:")
print(engineered.groupby(PITCH_TYPE_COL)["axis_differential"].median().sort_values().round(1))

print()
print("Split by pitcher handedness, FF/SI:")
print(engineered[engineered[PITCH_TYPE_COL].isin(["FF", "SI"])].groupby([PITCH_TYPE_COL, "p_throws"])["axis_differential"].median().round(1))

Median axis_differential (degrees) by pitch type:
pitch_type
FF     8.7
CU     9.3
KC    10.3
CH    12.4
SI    18.6
FS    20.6
FC    26.3
SL    27.6
SV    29.1
ST    31.1
Name: axis_differential, dtype: float64

Split by pitcher handedness, FF/SI:


pitch_type  p_throws
FF          L            9.3
            R            8.5
SI          L           16.5
            R           19.5
Name: axis_differential, dtype: float64


### Synopsis: matches known spin-efficiency patterns

Median gap ranges from 8.7 degrees (FF) to 31.1 degrees (ST), in the expected
order: four-seamers and curveballs (highest spin efficiency, closest to pure
Magnus) sit tightest; sinkers and splitters (real seam-shifted wake) and
cutters/sliders/sweepers (gyro-heavy, spin axis pointed more along the
direction of travel than perpendicular to it) sit loosest. Holds up split by
pitcher handedness too (FF: 8.5 RHP / 9.3 LHP; SI: 19.5 RHP / 16.5 LHP) -- the
pattern is not an artifact of one handedness dominating the pooled number.

## Leaderboard: top reliable pitcher-pitch-type-seasons

In [4]:
pitch_counts = (
    df.dropna(subset=["pitch_stuff_plus"])
    .groupby([PITCHER_COL, PITCH_TYPE_COL, SEASON_COL], observed=True)
    .size()
    .rename("n_pitches")
    .reset_index()
)

pitcher_stuff_plus = (
    df[df["stuff_plus_reliable"] == True]
    .drop_duplicates(subset=[PITCHER_COL, PITCH_TYPE_COL, SEASON_COL])
    .merge(pitch_counts, on=[PITCHER_COL, PITCH_TYPE_COL, SEASON_COL], how="left")
    [[PITCHER_COL, PLAYER_NAME_COL, PITCH_TYPE_COL, SEASON_COL, "n_pitches", "stuff_plus"]]
    .sort_values("stuff_plus", ascending=False)
    .reset_index(drop=True)
)

pitcher_stuff_plus.head(15)

,pitcher,player_name,pitch_type,game_year,n_pitches,stuff_plus
0,642207,"Williams, Devin",CH,2025,583,160.129488
1,642207,"Williams, Devin",CH,2022,620,159.462080
2,642207,"Williams, Devin",CH,2024,176,158.545722
3,661403,"Clase, Emmanuel",FC,2023,754,156.069734
4,642207,"Williams, Devin",CH,2023,548,154.037327
5,642207,"Williams, Devin",CH,2021,621,152.196854
6,669203,"Burnes, Corbin",FC,2024,1347,151.007240
7,669203,"Burnes, Corbin",FC,2021,1352,150.052747
8,661403,"Clase, Emmanuel",FC,2022,549,148.969521
9,666808,"Doval, Camilo",FC,2021,180,147.438676


In [5]:
top5_by_type = (
    pitcher_stuff_plus
    .sort_values([PITCH_TYPE_COL, "stuff_plus"], ascending=[True, False])
    .groupby(PITCH_TYPE_COL)
    .head(5)
    .reset_index(drop=True)
)
top5_by_type["rank"] = top5_by_type.groupby(PITCH_TYPE_COL).cumcount() + 1
top5_by_type[[PITCH_TYPE_COL, "rank", PLAYER_NAME_COL, SEASON_COL, "n_pitches", "stuff_plus"]]

,pitch_type,rank,player_name,game_year,n_pitches,stuff_plus
0,CH,1,"Williams, Devin",2025,583,160.129488
1,CH,2,"Williams, Devin",2022,620,159.462080
2,CH,3,"Williams, Devin",2024,176,158.545722
3,CH,4,"Williams, Devin",2023,548,154.037327
4,CH,5,"Williams, Devin",2021,621,152.196854
5,CU,1,"Glasnow, Tyler",2021,181,143.440612
6,CU,2,"Pérez, Cionel",2023,57,134.288339
7,CU,3,"Pressly, Ryan",2023,260,133.271120
8,CU,4,"Johnson, DJ",2021,20,132.943418
9,CU,5,"Maples, Dillon",2021,71,132.386423


## Diagnostic: does the retooled Stuff+ track bat-missing ability better?

The old PCA composite's own outcome-validation diagnostic (`docs/dev_log.md`'s
9/2 entry) found a real but weak, inconsistent relationship with CSW%
(called-strike-plus-whiff rate): pooled Spearman rho +0.078, and *negative*
point estimates for changeups and splitters. That result is part of what
motivated rereading FanGraphs' primer and retooling the weighting mechanism
in the first place -- a composite whose own "outlierness" axis only weakly
tracks the outcome it's supposed to predict is a sign the weights need to
come from somewhere other than raw variance.

Rerunning the identical diagnostic against the retooled Stuff+ (same CSW%
proxy, same >=100-pitch reliability bar, same 2023-2024 window) checks
whether training the weights against run value closed that gap.

In [6]:
outcome_cols = [PITCHER_COL, PITCH_TYPE_COL, SEASON_COL, "description"]
outcomes = pd.read_csv(DATA_PATH, usecols=outcome_cols)
outcomes = outcomes[
    outcomes[SEASON_COL].isin([2023, 2024]) & ~outcomes[PITCH_TYPE_COL].isin(JUNK_PITCH_TYPES)
].copy()
outcomes["csw"] = outcomes["description"].isin(["called_strike", "swinging_strike", "swinging_strike_blocked"])

csw_agg = (
    outcomes
    .groupby([PITCHER_COL, PITCH_TYPE_COL, SEASON_COL], observed=True)["csw"]
    .agg(csw_pct="mean", n_pitches_outcome="size")
    .reset_index()
)

validation = pitcher_stuff_plus.merge(csw_agg, on=[PITCHER_COL, PITCH_TYPE_COL, SEASON_COL], how="inner")
validation = validation[validation["n_pitches_outcome"] >= 100]
print(f"{len(validation):,} reliable pitcher-pitch_type-seasons with both a Stuff+ score and >=100 pitches of outcome data")

overall_rho, overall_p = spearmanr(validation["stuff_plus"], validation["csw_pct"])
print(f"\noverall: spearman(stuff_plus, csw_pct) = {overall_rho:+.3f} (p={overall_p:.2g}, n={len(validation):,})\n")

rows = []
for ptype, g in validation.groupby(PITCH_TYPE_COL):
    if len(g) < 30:
        continue
    rho, p = spearmanr(g["stuff_plus"], g["csw_pct"])
    rows.append({"pitch_type": ptype, "n": len(g), "spearman_rho": rho, "p_value": p})

pd.DataFrame(rows).sort_values("spearman_rho", ascending=False).reset_index(drop=True)

3,721 reliable pitcher-pitch_type-seasons with both a Stuff+ score and >=100 pitches of outcome data

overall: spearman(stuff_plus, csw_pct) = +0.136 (p=9.6e-17, n=3,721)



,pitch_type,n,spearman_rho,p_value
0,KC,77,0.221699,5.265106e-02
1,FC,323,0.198090,3.411500e-04
2,ST,305,0.179057,1.691444e-03
3,FF,988,0.170033,7.573495e-08
4,CH,450,0.168702,3.249617e-04
5,SL,618,0.149906,1.838655e-04
6,CU,269,0.131511,3.106251e-02
7,SI,572,0.093782,2.489879e-02
8,FS,102,0.035963,7.197057e-01


### Synopsis: a real improvement, not just a different shape

Pooled Spearman rho is +0.136 (p=9.6e-17, n=3,721) -- up from the old PCA
composite's +0.078 (p=2e-6) on the identical population. More telling than the
pooled number: every one of the nine pitch types now shows a positive point
estimate. The old composite had two negative ones -- changeups (-0.056) and
splitters (-0.141) -- exactly the off-speed types where "outlierness" relative
to other pitches of the same type was the shakiest proxy for bat-missing
ability. Under the retooled model, CH is +0.169 and FS is +0.036: still the
weakest of the nine (split-finger's reliable sample is thin, n=102), but no
longer pointing the wrong direction.

This is not a claim that the retool is "solved" -- +0.136 is still a modest
correlation, and CSW% is one proxy among several reasonable ones; whiff
rate, chase rate, and hard-hit rate are the natural next checks. But it is
a measured improvement on the exact diagnostic that originally flagged the
old design's weak spot, and it lines up with the larger jump already seen
in pitching.py's downstream Pitching+ blend (docs/dev_log.md's 9/4 entry:
stuff-alone R^2 0.02 -> 0.055, combined blend 0.08 -> 0.123).